In [1]:
# Use only 1 GPU if available
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # Disable progress bars

from chronos import BaseChronosPipeline, Chronos2Pipeline
from datetime import datetime
import torch
import os
from datetime import datetime
from ts_toolkit.utils.chronos_tune_input_generator import chronos_input_generator


torch.cuda.empty_cache()


## Can the model learn lagged copy paste or is it an architectural issue?
Experiment: Finetune Chronos on I.I.D. White Noise & or Stochastic Drift/Shocks
Measurement: Measure finetuned vs base performance


### Data Generation ###

In [2]:
DISTRIBUTIONS = {
    "Deterministic Trends": ['constant', 'linear', 'exponential', 'staircase'],
    "Harmonic & Periodic": ['sine', 'sawtooth', 'square', 'triangle', 'noise_patch', 'fourier'],
    "Transient Dynamics": ['damped_sine', 'growing_sine', 'chirp'],
    "Autocorrelated Processes": ['ar', 'ma', 'arma', 'sarima', 'garch', 'pink_noise', 'fractional_brownian', 'markov_chain'], 
    "I.I.D. White Noise": ['normal', 'uniform', 'poisson', 'binary', 'student_t', 'lognormal', 'laplace', 'cauchy', 'skew_normal'],
    "Stochastic Drift/Shocks": ['random_walk', 'piecewise_constant', 'gbm', 'intermittent', 'impulse', 'logistic'], 
}

In [3]:
# List of level 1 dataset types, e.g., ["univariate_lagged", "multivariate_lagged"]. If None, all types are used.
dataset_level1 =["multivariate_lagged"]
# List of level 2 dataset types, e.g., ["exponential", "noise_patch", "normal", "poisson", "random_walk"]. If None, all types are used.
# all but Autocorrelated Processes
dataset_level2 = DISTRIBUTIONS["I.I.D. White Noise"] + DISTRIBUTIONS["Stochastic Drift/Shocks"]

data_path = "/home/nafiseh/GitHub/timeseries-research/data/simpletime/datasets_100k"
train_inputs, validation_inputs = chronos_input_generator(data_path, dataset_level1, dataset_level2, val_size=0.1)


multivariate_lagged/normal: 100000it [00:18, 5297.85it/s]
multivariate_lagged/uniform: 100000it [00:18, 5316.79it/s]
multivariate_lagged/poisson: 100000it [00:19, 5237.90it/s]
multivariate_lagged/binary: 100000it [00:19, 5249.34it/s]
multivariate_lagged/student_t: 100000it [00:19, 5121.56it/s]
multivariate_lagged/lognormal: 100000it [00:18, 5290.23it/s]
multivariate_lagged/laplace: 100000it [00:18, 5286.37it/s]
multivariate_lagged/cauchy: 100000it [00:19, 5227.41it/s]
multivariate_lagged/skew_normal: 100000it [00:19, 5097.52it/s]
multivariate_lagged/random_walk: 100000it [00:20, 4832.83it/s]
multivariate_lagged/piecewise_constant: 100000it [00:19, 5223.05it/s]
multivariate_lagged/gbm: 100000it [00:19, 5008.42it/s]
multivariate_lagged/intermittent: 100000it [00:20, 4982.79it/s]
multivariate_lagged/impulse: 100000it [00:19, 5140.92it/s]
multivariate_lagged/logistic: 100000it [00:19, 5065.74it/s]
100%|██████████| 15/15 [04:52<00:00, 19.47s/it]



Prepared training inputs for 1350000 training time series and 150000 validation time series.

Sample time series:
  Target shape: (256,)


### Fine-Tuning API

The `fit` method accepts:
- `inputs`: Time series for fine-tuning (same format as predict_quantiles)
- `prediction_length`: Forecast horizon for fine-tuning
- `validation_inputs`: Optional validation data (same format as inputs)
- `learning_rate`: Optimizer learning rate (default: 1e-5)
- `num_steps`: Number of training steps (default: 1000)
- `batch_size`: Batch size for training (default: 256)

Returns a new pipeline with the fine-tuned model.

In [ ]:
output_dir = f"results/finetuned_models/chronos2/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
# Load the Chronos-2 pipeline
# GPU recommended for faster inference, but CPU is also supported using device_map="cpu"
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

print(f"Fine-tuning Chronos-2 model. Output directory: {output_dir}")

# Fine-tune the model
finetuned_pipeline = pipeline.fit(
    inputs=train_inputs,
    prediction_length=48,
    num_steps=100_000,
    learning_rate=1e-4,
    batch_size=256,
    logging_steps=100,
    # finetune_mode="lora",
    output_dir=output_dir,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    validation_inputs=validation_inputs,
    eval_steps=100
) 

/tmp/ipykernel_1023157/786704418.py:7: FutureWarning: Fine-tuning support is experimental and may be changed in future versions.
  finetuned_pipeline = pipeline.fit(
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/torch/backends/cuda/__init__.py:131: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return torch._C._get_cublas_allow_tf32()
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss,Validation Loss
100,2.173100,0.686566
200,1.072100,0.569956
300,1.033200,0.548758
400,0.879500,0.474603
500,0.916500,0.463065
600,0.881500,0.455435
700,0.801500,0.433951
800,0.810900,0.406830
900,0.813400,0.430632
1000,0.806200,0.416055


In [ ]:
output_dir

'results/finetuned_models/chronos2/20260211_233552'

In [ ]:
break

Results written in: 'results/finetuned_models/chronos2/20260212_113124'

In [ ]:
from ts_toolkit.pipeline.visualization import finetuning_curves

finetuning_curves(output_dir)

In [ ]:
from ts_toolkit.pipeline.runner import TsPipeline
from ts_toolkit.pipeline.utils import load_config
from glob import glob
import os

ft_experiment = "CopyPaste"
output_paths = {}

for multivariate in [True, False]:
    multivariate_str = "Multivariate" if multivariate else "Univariate"
    model_repo = f"results/finetuned_models/chronos2/20260211_233552"
    config_dict = {
        "experiment_name": f"Chronos2-FineTuned-{ft_experiment}-{multivariate_str}-SimpleTime",

        "paths": {
            "storage_path": "data/simpletime/datasets_100",
            "output_path": "results"
        },
        "forecaster": {
            "name": "MedianEnsemble",
            "max_length": 512,
            "models": [
                {
                    "name": "Chronos2",
                    "params": {
                        "repo_id": f"{model_repo}/finetuned-ckpt",
                        "batch_size": 64
                    },
                }
            ],
            "alias": f"Chronos2-FineTuned-{ft_experiment}-{multivariate_str}-SimpleTime"
        },
        "evaluation": {
            "to_univariate": not multivariate,
            "show_plot": False,
            "eval_substr": "dim0"
        }
    }
    
    config = load_config(config_dict=config_dict)
    # --- Set up the evaluation pipeline ---
    pipeline = TsPipeline(config)
    pipeline.run(skip_existing=True, num_plots=1)

    pipeline.get_results()
    output_paths[f"Chronos2 (Finetuned - {ft_experiment}) - {multivariate_str}"] = pipeline.output_path


/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


Evaluating Datasets:   0%|          | 0/108 [00:00<?, ?it/s]

100it [00:00, 959.76it/s]
100it [00:00, 962.38it/s]
100it [00:00, 924.72it/s]
100it [00:00, 990.97it/s]
100it [00:00, 1029.34it/s]
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/gluonts/ev/aggregations.py:123: RuntimeWarning: invalid value encountered in divide
  return self.partial_result / self.n
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/home/nafiseh/GitHub/timeseri

Evaluating Datasets:   0%|          | 0/108 [00:00<?, ?it/s]

100it [00:00, 854.18it/s]
100it [00:00, 932.89it/s]
100it [00:00, 891.81it/s]
100it [00:00, 786.64it/s]
100it [00:00, 848.57it/s]
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/gluonts/ev/aggregations.py:123: RuntimeWarning: invalid value encountered in divide
  return self.partial_result / self.n
100it [00:00, 918.54it/s]
100it [00:00, 868.27it/s]
100it [00:00, 840.35it/s]
100it [00:00, 898.55it/s]
100it [00:00, 929.64it/s]
100it [00:00, 874.58it/s]
100it [00:00, 864.97it/s]
100it [00:00, 864.47it/s]
100it [00:00, 895.84it/s]
100it [00:00, 855.44it/s]
100it [00:00, 838.27it/s]
100it [00:00, 948.17it/s]
100it [00:00, 783.32it/s]
100it [00:00, 890.48it/s]
100it [00:00, 926.31it/s]
100it [00:00, 813.88it/s]
100it [00:00, 846.98it/s]
100it [00:00, 947.51it/s]
100it [00:00, 907.14it/s]
100it [00:00, 878.55it/s]
100it [00:00, 912.38it/s]
100it [00:00, 764.54it/s]
100it [00:00, 897.41it/s]
100it [00:00, 882.60it/s]
100it [00:00, 782.77it/s]
100it [00:00, 865.40it

In [ ]:
output_paths

{'Chronos2 (Finetuned - CopyPaste) - Multivariate': '/home/nafiseh/GitHub/timeseries-research/results/Chronos2-FineTuned-CopyPaste-Multivariate-SimpleTime_20260212-093125',
 'Chronos2 (Finetuned - CopyPaste) - Univariate': '/home/nafiseh/GitHub/timeseries-research/results/Chronos2-FineTuned-CopyPaste-Univariate-SimpleTime_20260212-093609'}

10k: {'Chronos2 (Finetuned - CopyPaste) - Multivariate': '/home/nafiseh/GitHub/timeseries-research/results/Chronos2-FineTuned-CopyPaste-Multivariate-SimpleTime_20260211-154609',
 'Chronos2 (Finetuned - CopyPaste) - Univariate': '/home/nafiseh/GitHub/timeseries-research/results/Chronos2-FineTuned-CopyPaste-Univariate-SimpleTime_20260211-155059'}
100k: 